# Probability & Statistics for Machine Learning

Machine learning is fundamentally about **making predictions under uncertainty**.
Probability gives us the language for uncertainty, and statistics gives us the tools
to learn from data.

This notebook builds your intuition with simulations and plots — you'll *see* the math in action.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12

---
## 1. Probability Basics — Quantifying Uncertainty

**Probability** is a number between 0 and 1 that measures how likely an event is.

- $P(A) = 0$ → impossible
- $P(A) = 1$ → certain
- $P(A) = 0.5$ → equally likely to happen or not

**Key rules:**
- **Complement:** $P(\text{not } A) = 1 - P(A)$
- **Union:** $P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$
- **Independent events:** $P(A \text{ and } B) = P(A) \times P(B)$

Let's verify these by simulation.

In [ ]:
n_flips = 100_000
flips = np.random.choice(['H', 'T'], size=n_flips)
p_heads = np.mean(flips == 'H')
print(f"Simulated {n_flips:,} coin flips")
print(f"P(Heads) = {p_heads:.4f}  (theory: 0.5)")
print(f"P(Tails) = {1-p_heads:.4f}  (complement)")

n_rolls = 100_000
die1 = np.random.randint(1, 7, n_rolls)
die2 = np.random.randint(1, 7, n_rolls)

p_six_d1 = np.mean(die1 == 6)
p_six_d2 = np.mean(die2 == 6)
p_both_six = np.mean((die1 == 6) & (die2 == 6))
p_at_least_one_six = np.mean((die1 == 6) | (die2 == 6))

print(f"\nTwo dice, {n_rolls:,} rolls:")
print(f"P(die1=6) = {p_six_d1:.4f}  (theory: {1/6:.4f})")
print(f"P(both=6) = {p_both_six:.4f}  (theory: {1/36:.4f})")
print(f"P(at least one 6) = {p_at_least_one_six:.4f}  (theory: {11/36:.4f})")

In [ ]:
cumulative_p = np.cumsum(flips == 'H') / np.arange(1, n_flips + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(cumulative_p[:5000], 'b-', alpha=0.7)
ax.axhline(0.5, color='red', linestyle='--', label='True P(H) = 0.5')
ax.set_xlabel('Number of flips')
ax.set_ylabel('Running P(Heads)')
ax.set_title('Law of Large Numbers:\nP(Heads) converges to 0.5')
ax.legend()

ax = axes[1]
sums = die1 + die2
values, counts = np.unique(sums, return_counts=True)
ax.bar(values, counts / n_rolls, color='steelblue', edgecolor='black')
ax.set_xlabel('Sum of two dice')
ax.set_ylabel('Probability')
ax.set_title('Distribution of Sum of Two Dice')
ax.set_xticks(range(2, 13))

plt.tight_layout()
plt.show()

---
## 2. Conditional Probability & Bayes' Theorem

**Conditional probability** is the probability of an event **given that** another event has already occurred:

$$P(A | B) = \frac{P(A \text{ and } B)}{P(B)}$$

**Bayes' Theorem** lets you flip the condition — compute $P(A|B)$ from $P(B|A)$:

$$P(A | B) = \frac{P(B | A) \cdot P(A)}{P(B)}$$

**In ML:** The Naive Bayes classifier is literally this formula. Given features (B), what's the
probability of each class (A)?

In [ ]:
p_disease = 0.001
p_positive_given_disease = 0.99
p_positive_given_healthy = 0.05

p_positive = (p_positive_given_disease * p_disease +
              p_positive_given_healthy * (1 - p_disease))

p_disease_given_positive = (p_positive_given_disease * p_disease) / p_positive

print("Medical Test Example:")
print(f"  Disease prevalence:       P(D) = {p_disease:.3f} (1 in 1000)")
print(f"  Test sensitivity:         P(+|D) = {p_positive_given_disease:.2f}")
print(f"  False positive rate:      P(+|healthy) = {p_positive_given_healthy:.2f}")
print(f"")
print(f"  P(positive) = {p_positive:.4f}")
print(f"")
print(f"  P(disease | positive test) = {p_disease_given_positive:.4f} ({p_disease_given_positive*100:.2f}%)")
print(f"")
print(f"  Even with a 99% accurate test, a positive result only means")
print(f"  a ~{p_disease_given_positive*100:.0f}% chance of actually having the disease!")
print(f"  This is because the disease is rare (low prior probability).")

In [ ]:
population = 100_000
n_sick = int(population * p_disease)
n_healthy = population - n_sick

true_positive = int(n_sick * p_positive_given_disease)
false_negative = n_sick - true_positive
false_positive = int(n_healthy * p_positive_given_healthy)
true_negative = n_healthy - false_positive

fig, ax = plt.subplots(figsize=(8, 5))

categories = ['True Positive\n(sick, test +)', 'False Positive\n(healthy, test +)',
              'True Negative\n(healthy, test -)', 'False Negative\n(sick, test -)']
counts = [true_positive, false_positive, true_negative, false_negative]
colors = ['green', 'red', 'lightblue', 'orange']

bars = ax.bar(categories, counts, color=colors, edgecolor='black')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count:,}', ha='center', fontsize=11, fontweight='bold')

ax.set_title(f'Population of {population:,}: Why Most Positive Tests Are False Positives\n'
             f'P(disease|positive) = {true_positive}/{true_positive+false_positive} = '
             f'{true_positive/(true_positive+false_positive):.2%}')
ax.set_ylabel('Number of people')
plt.tight_layout()
plt.show()

---
## 3. Random Variables & Distributions

A **random variable** is a variable whose value is determined by a random process.
A **distribution** tells you all the possible values and how likely each one is.

### Discrete Distributions (countable outcomes)

| Distribution | What it models | Example |
|-------------|---------------|--------|
| **Bernoulli** | Single yes/no trial | One coin flip |
| **Binomial** | Number of successes in n trials | Heads in 10 flips |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
p = 0.7
ax.bar([0, 1], [1-p, p], color=['salmon', 'steelblue'], edgecolor='black', width=0.5)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Failure (0)', 'Success (1)'])
ax.set_ylabel('Probability')
ax.set_title(f'Bernoulli Distribution (p={p})')
ax.set_ylim(0, 1)

ax = axes[1]
n_trials = 20
for p_val, color in [(0.3, 'red'), (0.5, 'blue'), (0.7, 'green')]:
    k = np.arange(0, n_trials + 1)
    probs = stats.binom.pmf(k, n_trials, p_val)
    ax.plot(k, probs, 'o-', color=color, label=f'p={p_val}', markersize=5)

ax.set_xlabel('Number of successes (k)')
ax.set_ylabel('P(X = k)')
ax.set_title(f'Binomial Distribution (n={n_trials} trials)')
ax.legend()

plt.tight_layout()
plt.show()

### Continuous Distributions (any value in a range)

| Distribution | What it models | Example |
|-------------|---------------|--------|
| **Uniform** | All values equally likely | Random number between 0 and 1 |
| **Normal (Gaussian)** | Bell-shaped, centered at mean | Heights of people, measurement errors |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
samples = np.random.uniform(0, 1, 10_000)
ax.hist(samples, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
ax.axhline(1, color='red', lw=2, linestyle='--', label='PDF = 1 (theory)')
ax.set_xlabel('Value'); ax.set_ylabel('Density')
ax.set_title('Uniform Distribution [0, 1]')
ax.legend()

ax = axes[1]
x = np.linspace(-5, 5, 200)
for mu, sigma, color in [(0, 1, 'blue'), (0, 2, 'red'), (2, 0.5, 'green')]:
    pdf = stats.norm.pdf(x, mu, sigma)
    ax.plot(x, pdf, color=color, lw=2, label=f'μ={mu}, σ={sigma}')

ax.set_xlabel('Value'); ax.set_ylabel('Density')
ax.set_title('Normal (Gaussian) Distribution')
ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Mean, Variance, Standard Deviation — Summarizing Data

Three numbers that tell you almost everything about a distribution:

| Statistic | What it measures | Formula |
|-----------|-----------------|--------|
| **Mean** ($\mu$) | Center of the data | $\mu = \frac{1}{n}\sum x_i$ |
| **Variance** ($\sigma^2$) | How spread out (squared units) | $\sigma^2 = \frac{1}{n}\sum(x_i - \mu)^2$ |
| **Std Dev** ($\sigma$) | Spread (same units as data) | $\sigma = \sqrt{\sigma^2}$ |

In [ ]:
data_a = np.random.normal(loc=50, scale=5, size=1000)
data_b = np.random.normal(loc=50, scale=15, size=1000)

for name, data in [('Dataset A (tight)', data_a), ('Dataset B (spread)', data_b)]:
    print(f"{name}:")
    print(f"  Mean     = {np.mean(data):.2f}")
    print(f"  Variance = {np.var(data):.2f}")
    print(f"  Std Dev  = {np.std(data):.2f}")
    print()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data_a, bins=40, density=True, alpha=0.6, color='blue', label=f'A: μ=50, σ≈5')
ax.hist(data_b, bins=40, density=True, alpha=0.6, color='red', label=f'B: μ=50, σ≈15')
ax.axvline(50, color='black', lw=2, linestyle='--', label='Mean = 50')
ax.set_xlabel('Value'); ax.set_ylabel('Density')
ax.set_title('Same Mean, Different Spread (Variance)')
ax.legend()
plt.show()

---
## 5. The Normal Distribution & Central Limit Theorem

The **Normal (Gaussian) distribution** is the most important distribution in statistics and ML.
Why? Because of the **Central Limit Theorem (CLT)**:

> No matter what the original distribution looks like, the **average of many samples**
> from it will be approximately normal.

This is why the bell curve appears everywhere — test scores, measurement errors, stock returns.

Let's prove it by simulation: take samples from a **uniform** distribution (flat, not bell-shaped)
and watch their means form a bell curve.

In [ ]:
sample_sizes = [1, 2, 5, 30]
n_experiments = 10_000

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, n in zip(axes, sample_sizes):
    means = [np.mean(np.random.uniform(0, 1, n)) for _ in range(n_experiments)]

    ax.hist(means, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')

    if n > 1:
        mu = 0.5
        sigma = (1/12)**0.5 / n**0.5
        x = np.linspace(0, 1, 200)
        ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label='Normal fit')
        ax.legend(fontsize=9)

    ax.set_title(f'Mean of {n} uniform sample{"s" if n > 1 else ""}')
    ax.set_xlabel('Sample mean')

axes[0].set_ylabel('Density')
plt.suptitle('Central Limit Theorem: Means of uniform samples → Normal distribution',
             fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

### The 68-95-99.7 Rule

For a normal distribution:
- **68%** of data falls within 1 standard deviation of the mean
- **95%** within 2 standard deviations
- **99.7%** within 3 standard deviations

In [ ]:
x = np.linspace(-4, 4, 500)
pdf = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, pdf, 'k-', lw=2)

colors = ['#2196F3', '#4CAF50', '#FF9800']
sigmas = [1, 2, 3]
pcts = ['68%', '95%', '99.7%']

for sigma, color, pct in zip(sigmas, colors, pcts):
    mask = (x >= -sigma) & (x <= sigma)
    ax.fill_between(x[mask], pdf[mask], alpha=0.2, color=color, label=f'±{sigma}σ = {pct}')

ax.set_xlabel('Standard deviations from mean')
ax.set_ylabel('Density')
ax.set_title('The 68-95-99.7 Rule for Normal Distribution')
ax.legend(fontsize=12)
plt.show()

---
## 6. Correlation — How Variables Move Together

**Correlation** measures the linear relationship between two variables:

- $r = +1$: perfect positive (both go up together)
- $r = 0$: no linear relationship
- $r = -1$: perfect negative (one goes up, other goes down)

$$r = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \sum(y_i - \bar{y})^2}}$$

**In ML:** Correlation helps with feature selection. Highly correlated features carry redundant information.

In [ ]:
n = 200
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

correlations = [
    ('Strong positive\nr ≈ 0.95', 0.95),
    ('Weak positive\nr ≈ 0.4', 0.4),
    ('No correlation\nr ≈ 0', 0.0),
    ('Strong negative\nr ≈ -0.9', -0.9),
]

for ax, (title, target_r) in zip(axes, correlations):
    if target_r == 0:
        x_data = np.random.randn(n)
        y_data = np.random.randn(n)
    else:
        cov_matrix = [[1, target_r], [target_r, 1]]
        data = np.random.multivariate_normal([0, 0], cov_matrix, n)
        x_data, y_data = data[:, 0], data[:, 1]

    actual_r = np.corrcoef(x_data, y_data)[0, 1]
    ax.scatter(x_data, y_data, alpha=0.5, s=15)
    ax.set_title(f'{title}\n(actual: {actual_r:.2f})')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('Correlation: How Two Variables Move Together', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

### Correlation ≠ Causation

Just because two things are correlated doesn't mean one *causes* the other.
Ice cream sales and drowning rates are both correlated with hot weather — but
ice cream doesn't cause drowning!

---
## 7. Maximum Likelihood Estimation (MLE)

**MLE** answers the question: given the data I observed, what parameters make
this data **most likely** to have occurred?

**Example:** You flip a coin 100 times and get 73 heads. What's the most likely value of $p$
(the probability of heads)?

MLE says: the value of $p$ that maximizes the probability of seeing 73 heads in 100 flips.

$$L(p) = \binom{100}{73} p^{73} (1-p)^{27}$$

The maximum is at $\hat{p} = 73/100 = 0.73$ (the intuitive answer!).

In [ ]:
n_flips = 100
n_heads = 73

p_values = np.linspace(0.01, 0.99, 500)
likelihoods = stats.binom.pmf(n_heads, n_flips, p_values)

mle_p = p_values[np.argmax(likelihoods)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(p_values, likelihoods, 'b-', lw=2)
ax.axvline(mle_p, color='red', linestyle='--', lw=2, label=f'MLE: p̂ = {mle_p:.2f}')
ax.fill_between(p_values, likelihoods, alpha=0.2)
ax.set_xlabel('p (probability of heads)')
ax.set_ylabel('Likelihood: P(73 heads in 100 flips | p)')
ax.set_title('Maximum Likelihood Estimation\nWhich p makes our data most likely?')
ax.legend(fontsize=12)
plt.show()

print(f"MLE estimate: p̂ = {mle_p:.3f}")
print(f"Intuitive: 73/100 = {73/100:.3f}")

### MLE for a Normal Distribution

Given data, the MLE estimates for a Normal distribution are simply:
- $\hat{\mu} = \bar{x}$ (sample mean)
- $\hat{\sigma}^2 = \frac{1}{n}\sum(x_i - \bar{x})^2$ (sample variance)

MLE says: "the best-fitting Normal has its center at the average and its width matching the data's spread."

In [ ]:
true_mu, true_sigma = 5, 2
data = np.random.normal(true_mu, true_sigma, 200)

mle_mu = np.mean(data)
mle_sigma = np.std(data)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='black', label='Data')

x = np.linspace(data.min()-1, data.max()+1, 200)
ax.plot(x, stats.norm.pdf(x, mle_mu, mle_sigma), 'r-', lw=2,
        label=f'MLE fit: N({mle_mu:.2f}, {mle_sigma:.2f}²)')
ax.plot(x, stats.norm.pdf(x, true_mu, true_sigma), 'g--', lw=2,
        label=f'True: N({true_mu}, {true_sigma}²)')

ax.set_xlabel('Value'); ax.set_ylabel('Density')
ax.set_title('MLE fits a Normal distribution to data')
ax.legend()
plt.show()

---
## 8. Bias-Variance Tradeoff — The Fundamental Tension in ML

Every model has two sources of error:

- **Bias**: error from wrong assumptions (model too simple → underfitting)
- **Variance**: error from being too sensitive to training data (model too complex → overfitting)

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

The goal is to find the **sweet spot** — complex enough to capture the pattern,
simple enough to generalize.

In [ ]:
np.random.seed(42)
x_true = np.linspace(0, 1, 100)
y_true = np.sin(2 * np.pi * x_true)

n_points = 15
x_train = np.sort(np.random.uniform(0, 1, n_points))
y_train = np.sin(2 * np.pi * x_train) + np.random.randn(n_points) * 0.3

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
degrees = [1, 4, 15]
titles = ['Underfitting (degree 1)\nHigh Bias, Low Variance',
          'Good Fit (degree 4)\nBalanced Bias-Variance',
          'Overfitting (degree 15)\nLow Bias, High Variance']

for ax, degree, title in zip(axes, degrees, titles):
    coeffs = np.polyfit(x_train, y_train, degree)
    y_pred = np.polyval(coeffs, x_true)

    ax.plot(x_true, y_true, 'g--', lw=2, alpha=0.5, label='True function')
    ax.scatter(x_train, y_train, c='blue', s=50, zorder=5, label='Training data')
    ax.plot(x_true, y_pred, 'r-', lw=2, label=f'Polynomial degree {degree}')

    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

degrees_range = range(1, 16)
train_errors = []
test_errors = []

for d in degrees_range:
    coeffs = np.polyfit(x_train, y_train, d)
    y_pred_train = np.polyval(coeffs, x_train)
    y_pred_true = np.polyval(coeffs, x_true)

    train_errors.append(np.mean((y_train - y_pred_train)**2))
    test_errors.append(np.mean((y_true - y_pred_true)**2))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(degrees_range), train_errors, 'b-o', label='Training error', markersize=6)
ax.plot(list(degrees_range), test_errors, 'r-o', label='Test error', markersize=6)
ax.axvline(4, color='green', linestyle='--', alpha=0.5, label='Sweet spot')

ax.annotate('Underfitting\n(high bias)', xy=(1, test_errors[0]),
            xytext=(2, test_errors[0]+0.3), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='orange'))
ax.annotate('Overfitting\n(high variance)', xy=(14, test_errors[-1]),
            xytext=(11, test_errors[-1]+0.5), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Polynomial Degree (Model Complexity)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Bias-Variance Tradeoff: Training vs Test Error')
ax.legend()
ax.set_ylim(0, 2)
plt.show()

---
## Summary: Probability & Statistics in ML

| Concept | Where It Shows Up in ML |
|---------|------------------------|
| **Probability** | Classification outputs, uncertainty estimates |
| **Bayes' Theorem** | Naive Bayes, Bayesian neural networks |
| **Normal distribution** | Weight initialization, noise modeling, batch norm |
| **Mean/Variance** | Feature normalization, batch statistics |
| **CLT** | Why SGD works (averaging noisy gradients) |
| **Correlation** | Feature selection, multicollinearity detection |
| **MLE** | Logistic regression, fitting distributions |
| **Bias-Variance** | Model selection, regularization, cross-validation |